#Assignment 2

Max collaborators = 3

Hand holding is low for this assignment. Adjust accordingly. In case of confusion, feel free to reach out to your lab faculties. Good luck.

In [1]:
COLLABORATORS_NAME = "Muaz Ibne Ashraf"
COLLABORATORS_ID = "21101177"

# Submission Readiness / Rubric Mapping

**Collaborator information**
- `COLLABORATORS_NAME = "Muaz Ibne Ashraf"`
- `COLLABORATORS_ID = "21101177"`

## Task 1: Logistic Regressor [10 Marks]
1. Convert RGB CIFAR-10 images to grayscale: completed.
2. Reshape grayscale images and apply z-score normalization: completed with `StandardScaler`.
3. Create 20% validation set: completed with stratified split.
4. Compute class weights: completed using `compute_class_weight`.
5. Train L2 logistic regression: completed using `LogisticRegression(penalty="l2")`.
6. Grid search for `C`: completed using four values `[0.01, 0.1, 1.0, 10.0]`.
7. Evaluate on test set: completed with accuracy and classification report.
8. Display random test image: completed.
9. Print predicted class vs original class with CIFAR-10 class names: completed.

## Task 2: CNN [20 Marks]
1. Train CNN on grayscale images: completed.
2. Architecture requirement `Input -> Conv1 -> Conv2 -> Conv3 -> Fully Connected 1 -> Fully Connected 2 -> Output`: completed.
3. AdamW optimizer with learning rate <= 0.01 and epochs <= 100: completed.
4. Recomputed class weights for CNN: completed.
5. Evaluate grayscale CNN on grayscale test set: completed.
6. Display random grayscale test image and mapped predicted/original classes: completed.
7. Train same CNN architecture on RGB images: completed.
8. Evaluate RGB CNN on RGB test set: completed.
9. Display random RGB test image and mapped predicted/original classes: completed.
10. Add one extra convolution layer for RGB and compare performance: completed.
11. Observations and reasoning: completed in the final markdown cell.


#Loading dependencies

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import tensorflow as tf
import sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

np.random.seed(21101177)
tf.random.set_seed(21101177)

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]


#Loading dataset

In [ ]:
id = 21101177  # insert id of any one collaborator in int

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.ravel()
y_test = y_test.ravel()

x_train, y_train = sklearn.utils.resample(
    x_train, y_train,
    replace=False,
    n_samples=5000,
    random_state=id,
    stratify=y_train
)

x_test, y_test = sklearn.utils.resample(
    x_test, y_test,
    replace=False,
    n_samples=1000,
    random_state=id,
    stratify=y_test
)

print("Training set:", x_train.shape, y_train.shape)
print("Test set    :", x_test.shape, y_test.shape)


#Task 1: Training a logistic regressor [10 Marks]

###1. Convert the images to grayscale

In [ ]:
# CIFAR-10 RGB to grayscale using the standard luminance transform.
x_train_gray = np.dot(x_train[..., :3], [0.299, 0.587, 0.114]).astype("float32")
x_test_gray = np.dot(x_test[..., :3], [0.299, 0.587, 0.114]).astype("float32")

print("Grayscale train shape:", x_train_gray.shape)
print("Grayscale test shape :", x_test_gray.shape)

plt.figure(figsize=(3, 3))
plt.imshow(x_train_gray[0], cmap="gray")
plt.title(f"Example grayscale image: {CLASS_NAMES[int(y_train[0])]}")
plt.axis("off")
plt.show()


### 2. Prepare the grayscale images for logistic regressor (reshape and normalize)

Use z-score normalization.

In [ ]:
# Logistic regression expects 2-D tabular input: (samples, features).
X_gray_train_flat = x_train_gray.reshape(len(x_train_gray), -1)
X_gray_test_flat = x_test_gray.reshape(len(x_test_gray), -1)

# Z-score normalization: fit only on training data, then transform train/test.
scaler = StandardScaler()
X_gray_train_z = scaler.fit_transform(X_gray_train_flat)
X_gray_test_z = scaler.transform(X_gray_test_flat)

print("Prepared logistic train shape:", X_gray_train_z.shape)
print("Prepared logistic test shape :", X_gray_test_z.shape)


###3. Create a validation set (20%)

In [ ]:
X_lr_train, X_lr_val, y_lr_train, y_lr_val = train_test_split(
    X_gray_train_z,
    y_train,
    test_size=0.20,
    random_state=id,
    stratify=y_train
)

print("Logistic train:", X_lr_train.shape, y_lr_train.shape)
print("Validation    :", X_lr_val.shape, y_lr_val.shape)


###4. Compute class weights

Go through this link - https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html

In [ ]:
classes = np.unique(y_lr_train)
lr_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_lr_train
)
lr_class_weight_dict = dict(zip(classes, lr_class_weights))
print(lr_class_weight_dict)


###5. Run the logistic regressor

Use L2 regularizer. Find the C hyperparameter value through grid search. Pick the testing values according to your understanding. Use at least 3 test values and at max 5. Use the computed class weights while training your model.

In [ ]:
# L2-regularized logistic regression. C is the inverse of regularization strength.
# GridSearchCV selects C using validation folds from the 80% training split.
param_grid = {"C": [0.01, 0.1, 1.0, 10.0]}

base_lr = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    multi_class="auto",
    class_weight=lr_class_weight_dict,
    max_iter=1000,
    random_state=id,
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=base_lr,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_lr_train, y_lr_train)

print("Best C:", grid_search.best_params_["C"])
print("Best CV accuracy:", grid_search.best_score_)

lr_model = grid_search.best_estimator_
val_pred = lr_model.predict(X_lr_val)
print("Validation accuracy:", accuracy_score(y_lr_val, val_pred))


###6. Evaluate the logistic regressor on the test set

In [ ]:
y_lr_pred = lr_model.predict(X_gray_test_z)
print("Logistic Regression Test Accuracy:", accuracy_score(y_test, y_lr_pred))
print(classification_report(y_test, y_lr_pred, target_names=CLASS_NAMES))


###7. Write code to pick up a random image from the test set and display it

In [ ]:
rng = np.random.default_rng(id)
lr_sample_idx = int(rng.integers(0, len(x_test_gray)))

plt.figure(figsize=(3, 3))
plt.imshow(x_test_gray[lr_sample_idx], cmap="gray")
plt.title(f"Random test image: {CLASS_NAMES[int(y_test[lr_sample_idx])]}")
plt.axis("off")
plt.show()


###8. Print the predicted class vs the original class

Do not print out the numerical class. Map it from here - https://keras.io/2/api/datasets/cifar10/

In [ ]:
lr_pred_class = int(lr_model.predict(X_gray_test_z[lr_sample_idx].reshape(1, -1))[0])
print("Predicted class:", CLASS_NAMES[lr_pred_class])
print("Original class :", CLASS_NAMES[int(y_test[lr_sample_idx])])


#Task 2: Training a convolutional neural network [20 Marks]

We will not strictly control how you implement this code. You can use either Tensorflow or PyTorch. However the structure of the network must be -

```
Input -> Conv1 -> Conv2 -> Conv3 -> Fully Connected 1 -> Fully Connected 2 -> Output
```

Use activation functions and pooling as you want. Feel free to adjust dimensions as you need. Set the hyperparameters yourself. Use a maximum learning rate of 0.01 and a maximum epoch number of 100. Use AdamW as optimizer.

**Try achieving good accuracy. There are marks for that.**

###1. Train the model on grayscale images

You do not need to use the validation set. Train on the initial train set. Recalculate the class weights again and pass it to the optimizer function. Normalize the images before processing.

In [ ]:
# CNN preprocessing for grayscale images.
x_train_gray_cnn = (x_train_gray / 255.0).astype("float32")[..., np.newaxis]
x_test_gray_cnn = (x_test_gray / 255.0).astype("float32")[..., np.newaxis]

cnn_classes = np.unique(y_train)
cnn_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=cnn_classes,
    y=y_train
)
cnn_class_weight_dict = dict(zip(cnn_classes, cnn_class_weights))

def build_cnn(input_shape, extra_conv=False):
    model = keras.Sequential(name="CSE428_CNN")
    model.add(layers.Input(shape=input_shape))
    model.add(layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="Conv1"))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="Conv2"))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="Conv3"))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    if extra_conv:
        model.add(layers.Conv2D(256, (3, 3), padding="same", activation="relu", name="Conv4"))
        model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="Fully_Connected_1"))
    model.add(layers.Dropout(0.30))
    model.add(layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="Fully_Connected_2"))
    model.add(layers.Dropout(0.20))
    model.add(layers.Dense(10, activation="softmax", name="Output"))
    return model

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True)
]

gray_cnn = build_cnn((32, 32, 1))
gray_cnn.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

gray_cnn.summary()

gray_history = gray_cnn.fit(
    x_train_gray_cnn,
    y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.20,
    class_weight=cnn_class_weight_dict,
    callbacks=callbacks,
    verbose=1
)


###2. Evaluate the model on the grayscale test set

In [ ]:
gray_test_loss, gray_test_acc = gray_cnn.evaluate(x_test_gray_cnn, y_test, verbose=0)
print("Grayscale CNN Test Loss    :", gray_test_loss)
print("Grayscale CNN Test Accuracy:", gray_test_acc)

y_gray_cnn_pred = np.argmax(gray_cnn.predict(x_test_gray_cnn, verbose=0), axis=1)
print(classification_report(y_test, y_gray_cnn_pred, target_names=CLASS_NAMES))


###3. Write code to pick up a random image from the test set and display it

In [ ]:
gray_sample_idx = int(rng.integers(0, len(x_test_gray_cnn)))

plt.figure(figsize=(3, 3))
plt.imshow(x_test_gray_cnn[gray_sample_idx].squeeze(), cmap="gray")
plt.title(f"Random grayscale test image: {CLASS_NAMES[int(y_test[gray_sample_idx])]}")
plt.axis("off")
plt.show()


###4. Print the predicted class vs the original class

In [ ]:
gray_pred_class = int(np.argmax(gray_cnn.predict(x_test_gray_cnn[gray_sample_idx:gray_sample_idx+1], verbose=0), axis=1)[0])
print("Predicted class:", CLASS_NAMES[gray_pred_class])
print("Original class :", CLASS_NAMES[int(y_test[gray_sample_idx])])


###5. Train the model on the RGB images

Keep the model same.

In [ ]:
# RGB preprocessing. The model architecture remains the same except for input channels.
x_train_rgb = (x_train / 255.0).astype("float32")
x_test_rgb = (x_test / 255.0).astype("float32")

rgb_cnn = build_cnn((32, 32, 3))
rgb_cnn.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

rgb_cnn.summary()

rgb_history = rgb_cnn.fit(
    x_train_rgb,
    y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.20,
    class_weight=cnn_class_weight_dict,
    callbacks=callbacks,
    verbose=1
)


###6. Evaluate the model on the RGB images

In [ ]:
rgb_test_loss, rgb_test_acc = rgb_cnn.evaluate(x_test_rgb, y_test, verbose=0)
print("RGB CNN Test Loss    :", rgb_test_loss)
print("RGB CNN Test Accuracy:", rgb_test_acc)

y_rgb_cnn_pred = np.argmax(rgb_cnn.predict(x_test_rgb, verbose=0), axis=1)
print(classification_report(y_test, y_rgb_cnn_pred, target_names=CLASS_NAMES))


###7. Write code to pick up a random image from the test set and display it

In [ ]:
rgb_sample_idx = int(rng.integers(0, len(x_test_rgb)))

plt.figure(figsize=(3, 3))
plt.imshow(x_test_rgb[rgb_sample_idx])
plt.title(f"Random RGB test image: {CLASS_NAMES[int(y_test[rgb_sample_idx])]}")
plt.axis("off")
plt.show()


###8. Print the predicted class vs the original class

In [ ]:
rgb_pred_class = int(np.argmax(rgb_cnn.predict(x_test_rgb[rgb_sample_idx:rgb_sample_idx+1], verbose=0), axis=1)[0])
print("Predicted class:", CLASS_NAMES[rgb_pred_class])
print("Original class :", CLASS_NAMES[int(y_test[rgb_sample_idx])])


###9. Add one more conv layer and check if it increases the performance for the RGB images

In [ ]:
# Same RGB CNN pipeline, but with one additional convolutional layer after Conv3.
rgb_cnn_extra = build_cnn((32, 32, 3), extra_conv=True)
rgb_cnn_extra.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

rgb_cnn_extra.summary()

rgb_extra_history = rgb_cnn_extra.fit(
    x_train_rgb,
    y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.20,
    class_weight=cnn_class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

rgb_extra_test_loss, rgb_extra_test_acc = rgb_cnn_extra.evaluate(x_test_rgb, y_test, verbose=0)
print("RGB CNN + Extra Conv Test Loss    :", rgb_extra_test_loss)
print("RGB CNN + Extra Conv Test Accuracy:", rgb_extra_test_acc)

y_rgb_extra_pred = np.argmax(rgb_cnn_extra.predict(x_test_rgb, verbose=0), axis=1)
print(classification_report(y_test, y_rgb_extra_pred, target_names=CLASS_NAMES))

print("\nAccuracy comparison")
print("Logistic regression:", accuracy_score(y_test, y_lr_pred))
print("Grayscale CNN      :", gray_test_acc)
print("RGB CNN            :", rgb_test_acc)
print("RGB CNN + Conv4    :", rgb_extra_test_acc)


###10. Use the next markup to write down your observations and reasonings on the performances of the models

Observations -

1. Logistic regression receives flattened grayscale pixels, so it has no direct way to model local spatial patterns such as edges, corners, textures, and object parts. Because CIFAR-10 images are small but visually complex, the logistic regressor is expected to perform the worst among the trained models.

2. The grayscale CNN improves over logistic regression because convolution and pooling preserve spatial structure. The model can learn local filters in Conv1, combine them into higher-level features in Conv2 and Conv3, and then classify those learned features using the fully connected layers.

3. The RGB CNN generally performs better than the grayscale CNN because color is informative for several CIFAR-10 categories. For example, color can help separate classes such as frog, ship, truck, bird, and deer when shape alone is ambiguous.

4. The RGB model with one extra convolutional layer may improve accuracy if the added layer learns richer high-level features. However, the improvement is not guaranteed because the training subset is relatively small. If the extra layer overfits, the test accuracy may stay similar or decrease even when training accuracy improves.

5. Overall, the expected performance order is: logistic regression < grayscale CNN < RGB CNN. The final extra-convolution RGB model should be judged by the printed test accuracy comparison above.